# Generate Task 2 embeddings in Google Colab

Run this in **Google Colab** (colab.research.google.com). Colab has open internet,
so the two models download fine (no office-proxy SSL error). It then generates the
embeddings and pushes the small `.npy` files back to your GitHub repo. Your office
laptop just does `git pull` afterward and runs everything offline.

**Runtime → Run all**, and paste your GitHub token when asked.


## 1. Clone your repo (paste your GitHub token when prompted)


In [ ]:
from getpass import getpass
USER = "sanjaykumarpushadapu"
REPO = "mtech"
TOKEN = getpass("Paste your GitHub token (it stays hidden): ").strip()
!git clone https://{TOKEN}@github.com/{USER}/{REPO}.git
%cd {REPO}/2026-2027-Sem1/AIMLCZG521-ConversationalAI/ass-1
print("Now in:", __import__("os").getcwd())


## 2. Install the libraries


In [ ]:
!pip -q install sentence-transformers pandas numpy


## 3. Generate the embeddings

Loads the corpus/queries from the repo's `data/raw/` (same fixed order the notebook
uses), embeds documents **and** queries with both models, and saves the 5 result
files into `data/results/`.


In [ ]:
import os, json, time
import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer

RAW, OUT = "data/raw", "data/results"
os.makedirs(OUT, exist_ok=True)

corpus  = pd.read_json(f"{RAW}/corpus.jsonl",  lines=True)
queries = pd.read_json(f"{RAW}/queries.jsonl", lines=True)
qrels   = pd.read_csv(f"{RAW}/qrels.tsv", sep="\t")
corpus["doc_id"]    = corpus["doc_id"].astype(str)
queries["query_id"] = queries["query_id"].astype(str)
qrels["query_id"]   = qrels["query_id"].astype(str)
queries = queries[queries["query_id"].isin(set(qrels["query_id"]))].reset_index(drop=True)
print(f"corpus={len(corpus)}  queries={len(queries)}")

MODELS = {
    "distilbert-base-uncased": "distilbert-base-uncased",
    "BAAI/bge-large-en-v1.5":  "BAAI__bge-large-en-v1.5",
}
timings = {}
for name, safe in MODELS.items():
    print(f"\n=== {name} ===")
    model = SentenceTransformer(name)          # downloads on first use (fine in Colab)
    t0 = time.time()
    doc_emb = model.encode(corpus["text"].tolist(), show_progress_bar=True)
    timings[name] = round(time.time() - t0, 2)
    qry_emb = model.encode(queries["text"].tolist(), show_progress_bar=True)
    np.save(f"{OUT}/embeddings_{safe}.npy", doc_emb)
    np.save(f"{OUT}/query_embeddings_{safe}.npy", qry_emb)
    print(f"saved docs={doc_emb.shape} queries={qry_emb.shape} corpus_time={timings[name]}s")

json.dump(timings, open(f"{OUT}/task2_timings.json", "w"), indent=2)
print("\nAll embeddings saved:", timings)


## 4. Push the embeddings back to your repo


In [ ]:
!git config user.email "sanjaykumar.pushadapu@gmail.com"
!git config user.name "Sanjay Kumar Pushadapu"
!git add data/results/embeddings_*.npy data/results/query_embeddings_*.npy data/results/task2_timings.json
!git commit -m "Add Task 2 embeddings (generated in Google Colab)"
!git push
print("\nDone. On your office laptop run:  git pull   -- then Tasks 2-3 run offline.")
